# 🔧 Notebook 2: Data Preprocessing

**Môn học:** Học Máy (CO3001) — Học kỳ I (2025-2026)  
**Nhóm:** 13

### Mục tiêu
- Xử lý missing values (Mean / Median / KNN Imputer)
- Encoding categorical features (One-Hot / Label / Ordinal)
- Feature scaling (Standard / MinMax / Robust)
- So sánh các cấu hình preprocessing bằng Logistic Regression probe


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns, os, urllib.request, warnings, joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

for d in ['../data', 'data', '../features', 'features', '../models', 'models',
          '../reports/figures', 'reports/figures']:
    os.makedirs(d, exist_ok=True)

RANDOM_STATE = 42
DATA_DIR     = '../data'    if os.path.isdir('../data')    else 'data'
FEAT_DIR     = '../features' if os.path.isdir('../features') else 'features'
MOD_DIR      = '../models'  if os.path.isdir('../models')  else 'models'
FIG_DIR      = '../reports/figures' if os.path.isdir('../reports') else 'reports/figures'
print('✅ Libraries loaded!')


## 2. Load Dataset

In [ ]:
COLUMNS = ['age','workclass','fnlwgt','education','education_num',
           'marital_status','occupation','relationship','race','sex',
           'capital_gain','capital_loss','hours_per_week','native_country','income']
TRAIN_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
TEST_URL  = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"
TRAIN_PATH = os.path.join(DATA_DIR, 'adult_train.csv')
TEST_PATH  = os.path.join(DATA_DIR, 'adult_test.csv')

if not os.path.exists(TRAIN_PATH):
    print('Downloading...')
    urllib.request.urlretrieve(TRAIN_URL, TRAIN_PATH)
    urllib.request.urlretrieve(TEST_URL,  TEST_PATH)

df_train = pd.read_csv(TRAIN_PATH, names=COLUMNS, sep=r',\s*', engine='python', na_values='?')
df_test  = pd.read_csv(TEST_PATH,  names=COLUMNS, sep=r',\s*', engine='python', na_values='?', skiprows=1)
df_full  = pd.concat([df_train, df_test], ignore_index=True)
df_full['income'] = df_full['income'].str.replace('.', '', regex=False).str.strip()

df = df_full.drop_duplicates().reset_index(drop=True)
print(f'✅ Loaded: {df.shape[0]:,} mẫu × {df.shape[1]} cột')
print(f'Missing: {df.isnull().sum().sum()} giá trị trong {df.isnull().any().sum()} cột')


## 3. Tách Features & Target + Train-Test Split

In [ ]:
numeric_features     = ['age','fnlwgt','education_num','capital_gain','capital_loss','hours_per_week']
categorical_features = ['workclass','education','marital_status','occupation',
                        'relationship','race','sex','native_country']

X = df.drop(columns=['income'])
y = (df['income'] == '>50K').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Class ratio (train): >50K = {y_train.mean():.3f}')


## 4. Định Nghĩa Preprocessing Pipeline (Có Thể Cấu Hình)

In [ ]:
def build_preprocessor(impute_num='median', encoding='onehot', scaling='standard'):
    """
    Xây dựng preprocessing pipeline có thể cấu hình.
    impute_num : 'mean' | 'median' | 'knn'
    encoding   : 'onehot' | 'label'
    scaling    : 'standard' | 'minmax' | 'robust'
    """
    # Imputer
    if impute_num == 'knn':
        num_imputer = KNNImputer(n_neighbors=5)
    else:
        num_imputer = SimpleImputer(strategy=impute_num)
    cat_imputer = SimpleImputer(strategy='most_frequent')

    # Scaler
    scalers = {'standard': StandardScaler(), 'minmax': MinMaxScaler(), 'robust': RobustScaler()}
    scaler  = scalers[scaling]

    # Encoder
    if encoding == 'onehot':
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    else:  # label / ordinal
        encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

    numeric_pipeline    = Pipeline([('imputer', num_imputer), ('scaler', scaler)])
    categorical_pipeline = Pipeline([('imputer', cat_imputer), ('encoder', encoder)])

    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline,     numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ])
    return preprocessor

print('✅ build_preprocessor() sẵn sàng!')
print('   Tham số: impute_num (mean|median|knn) | encoding (onehot|label) | scaling (standard|minmax|robust)')


## 5. So Sánh Các Cấu Hình Preprocessing

In [ ]:
PREP_CONFIGS = {
    'Mean + OneHot + Standard' : dict(impute_num='mean',   encoding='onehot', scaling='standard'),
    'Median + OneHot + MinMax' : dict(impute_num='median', encoding='onehot', scaling='minmax'),
    'Median + Label + Robust'  : dict(impute_num='median', encoding='label',  scaling='robust'),
    'KNN + OneHot + Standard'  : dict(impute_num='knn',    encoding='onehot', scaling='standard'),
}

lr_probe = LogisticRegression(max_iter=500, random_state=RANDOM_STATE, class_weight='balanced')

prep_results = {}
for name, cfg in PREP_CONFIGS.items():
    prep = build_preprocessor(**cfg)
    X_tr_p = prep.fit_transform(X_train, y_train)
    X_te_p = prep.transform(X_test)
    lr_probe.fit(X_tr_p, y_train)
    acc = lr_probe.score(X_te_p, y_test)
    prep_results[name] = {'accuracy': acc, 'n_features': X_tr_p.shape[1]}
    print(f'[{name}]  accuracy={acc:.4f}  features={X_tr_p.shape[1]}')

best_cfg_name = max(prep_results, key=lambda k: prep_results[k]['accuracy'])
print(f'\n✅ Cấu hình tốt nhất: {best_cfg_name}')


## 6. Áp Dụng Best Config & Lưu Features

In [ ]:
# Dùng cấu hình tốt nhất
preprocessor = build_preprocessor(**PREP_CONFIGS[best_cfg_name])

X_train_proc = preprocessor.fit_transform(X_train, y_train)
X_test_proc  = preprocessor.transform(X_test)

print(f'Shape sau preprocessing — Train: {X_train_proc.shape}, Test: {X_test_proc.shape}')

# Lưu
np.save(os.path.join(FEAT_DIR, 'X_train_preprocessed.npy'), X_train_proc)
np.save(os.path.join(FEAT_DIR, 'X_test_preprocessed.npy'),  X_test_proc)
np.save(os.path.join(FEAT_DIR, 'y_train.npy'), y_train.values)
np.save(os.path.join(FEAT_DIR, 'y_test.npy'),  y_test.values)
joblib.dump(preprocessor, os.path.join(MOD_DIR, 'preprocessor.pkl'))
print('✅ Đã lưu features và preprocessor!')


## 7. Visualize Trước / Sau Preprocessing

In [ ]:
# Phân phối trước/sau scaling cho 'age'
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(X_train['age'].dropna(), bins=35, color='salmon', edgecolor='white', alpha=0.8)
axes[0].set_title('Trước Scaling: age', fontweight='bold')
axes[0].set_xlabel('age (original)')

axes[1].hist(X_train_proc[:, 0], bins=35, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('Sau Standard Scaling: age', fontweight='bold')
axes[1].set_xlabel('age (scaled)')

plt.suptitle('Before vs After Preprocessing', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/before_after_scaling.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# So sánh các config
fig, ax = plt.subplots(figsize=(10, 5))
names  = list(prep_results.keys())
accs   = [prep_results[n]['accuracy'] for n in names]
colors = ['#27ae60' if n == best_cfg_name else '#3498db' for n in names]
bars   = ax.barh(names, accs, color=colors, edgecolor='white', alpha=0.85)
for bar, v in zip(bars, accs):
    ax.text(v + 0.001, bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=10)
ax.set_xlabel('Accuracy (LR probe)')
ax.set_title('So sánh Preprocessing Configs — LR Probe Accuracy', fontsize=13, fontweight='bold')
ax.set_xlim(0.7, 0.9)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/preprocessing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Preprocessing hoàn thành! Chạy 03_Traditional_ML tiếp theo.')
